# FreshCart ELT — classroom-based practice notebook
The first calculation is an independent local cross-check. Live SQL cells are unexecuted until run in your own Snowflake account. Do not claim the local result is a Snowflake output.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "elt").is_dir(): root = root.parent
sys.path.insert(0,str(root))
from elt.local_reference import compute_baseline
import pandas as pd
summary, gold = compute_baseline()
print(summary)
display(pd.DataFrame(gold))

{'bronze': 8, 'clean': 6, 'quarantine': 2, 'dedup': 4, 'revenue_inr': '7785.00', 'points': 126, 'rejected_percent': 25.0}


,order_id,tier,amount_inr,loyalty_points
0,ORD-1001,Silver,1250.00,12
1,ORD-1002,Silver,3536.00,35
2,ORD-1003,Platinum,2100.00,63
3,ORD-1005,Gold,899.00,16


## B1 — live Snowflake setup
Use an empty dedicated database. For Colab, set a supported noninteractive authentication method (never expose a key or token); externalbrowser is intended for a local desktop. Load secrets securely. Run from the repo root.

In [ ]:
import os, re
from dotenv import load_dotenv
import snowflake.connector
from elt import classroom_sql as fc
load_dotenv(root / ".env")
# Use an isolated assignment database; no DROP commands in setup.
db=os.getenv("SNOWFLAKE_DATABASE", "FRESHCART_DE_LAB")
wh=os.getenv("SNOWFLAKE_WAREHOUSE", "FRESHCART_WH")
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", db) and re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", wh)
assert db.upper() == "FRESHCART_DE_LAB", "Select a fresh isolated assignment database"
conf=dict(account=os.environ["SNOWFLAKE_ACCOUNT"],user=os.environ["SNOWFLAKE_USER"],authenticator=os.getenv("SNOWFLAKE_AUTHENTICATOR","externalbrowser"))
if os.getenv("SNOWFLAKE_ROLE"): conf["role"]=os.environ["SNOWFLAKE_ROLE"]
if os.getenv("SNOWFLAKE_PASSWORD"): conf["password"]=os.environ["SNOWFLAKE_PASSWORD"]
if os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE"): conf["private_key_file"]=os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]
fc.conn=snowflake.connector.connect(**conf)
fc.LANDING_DIR=str(root / "landing")
fc.run_sql(fc.SQL_00_SETUP.replace("FRESHCART_WH",wh).replace("CREATE DATABASE IF NOT EXISTS FRESHCART;",f"CREATE DATABASE IF NOT EXISTS {db};").replace("USE DATABASE FRESHCART;",f"USE DATABASE {db};"))
assert fc.query("SELECT COUNT(*) FROM bronze.orders_raw")[0][0]==0, "Baseline requires empty project tables"
fc.run_pipeline()

In [ ]:
for table in ["bronze.orders_raw","silver.orders_clean","quarantine.orders","silver.orders_dedup"]:
    print(table,fc.query(f"SELECT COUNT(*) FROM {table}")[0][0])
display(fc.show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue, SUM(loyalty_points) AS points FROM gold.fct_orders"))

## B2 — trace the records
Inspect SQL in classroom_sql.py; write the explanation and arithmetic in your own words.

In [ ]:
display(fc.show("SELECT source,raw FROM bronze.orders_raw WHERE COALESCE(raw:order_id::STRING,raw:orderId::STRING)='ORD-1003'"))
display(fc.show("SELECT * FROM silver.orders_checked WHERE order_id='ORD-1003'"))
display(fc.show("SELECT order_id, tier, amount, fx_rate, amount_inr, loyalty_points FROM gold.fct_orders WHERE order_id IN ('ORD-1001','ORD-1003','ORD-1005')"))

## B3 — your validation change
Edit SQL_10_ORDERS_CLEAN in a new cell: insert the additional IFF in ARRAY_CONSTRUCT_COMPACT, then run the pipeline and compare current checked reasons and historical quarantine. The original notebook is preserved in classroom_sql.py.

In [ ]:
display(fc.show("SELECT order_id,reasons FROM silver.orders_checked WHERE order_id='ORD-1006'"))
# TODO(student): add validation, rerun, capture reasons and gold totals.

## B4 — prediction checkpoint
Write your own five predictions before adding the CRM event. Do not run this experiment until you have recorded them. Capture gold hashes and timestamps before and after; then rerun unchanged.

In [ ]:
# Capture BEFORE adding the B4 event
before=fc.show("SELECT order_id,tier,loyalty_points,row_hash,published_at FROM gold.fct_orders ORDER BY order_id")
display(before)
# TODO(student): add CRM event; run; query customer history and compare before/after.

## B5 — data quality experiment
Choose a sixth check and cause a real failure. Suggested future-date check: COUNT_IF(order_date > CURRENT_DATE()) in silver.orders_dedup. The classroom FX date spine ends at CURRENT_DATE, so a future record may also trigger the existing missing-FX check. Clean up only the uniquely named experimental rows in your isolated database, including the staged data if reloaded, and rerun.

In [ ]:
# TODO(student): add sixth check, inject unique mock order, demonstrate failure, remove and verify recovery.
# fc.conn.close()